# TanggapMakna — FINAL ONE-TIME BLIND EVALUATION

## V3 Fusion v1.1 — Locked Final Test

Notebook ini adalah **evaluasi final AI**, bukan notebook tuning.

Urutan wajib:

`Inference Input (tanpa label)`
→ `V3A + V2 + Fusion v1.1`
→ **SAVE / FREEZE PREDICTIONS**
→ baru buka `Locked Ground Truth`
→ hitung final metrics.

### Aturan keras

- V2 SRL: **LOCKED**
- V3A Event Classifier baseline: **LOCKED**
- Fusion v1.1: **LOCKED**
- Ground Truth: **LOCKED**
- Tidak ada threshold tuning.
- Tidak ada regex baru.
- Tidak ada perubahan model/rule berdasarkan hasil 60 blind reports ini.
- Jika hasil final memiliki kelemahan, catat sebagai limitation / future iteration.

Notebook ini sengaja memisahkan **inference** dan **evaluation** agar label ground truth
tidak tersedia pada saat prediksi dibuat.

## 0. File yang diperlukan

### Model — sudah ada di Google Drive

Folder yang diasumsikan:

`MyDrive/TanggapMakna/models/`

berisi:

- `TanggapMakna_IndoBERT_V2.zip`
- `TanggapMakna_V3A_EventClassifier.zip`

### File blind test

**Tahap inference terlebih dahulu:**
- `TanggapMakna_Blind_Inference_Input_v1.0.csv`

File ini hanya memiliki:
- `Report_ID`
- `Report_Text`

**Ground truth JANGAN di-upload sampai predictions sudah FROZEN:**
- `TanggapMakna_Blind_Ground_Truth_LOCKED_v1.0.csv`

## 1. Setup

In [ ]:
!nvidia-smi
!pip install -q transformers pandas scikit-learn

import os
import re
import json
import math
import shutil
import zipfile
import hashlib
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU belum aktif. Pilih Runtime > Change runtime type > GPU."
    )

print("=" * 80)
print("TANGGAPMAKNA — FINAL ONE-TIME BLIND EVALUATION")
print("=" * 80)
print("Device :", DEVICE)
print("GPU    :", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

## 2. Mount Google Drive dan ambil dua checkpoint LOCKED

Tidak ada upload model melalui browser.

Notebook menyalin ZIP dari Drive ke `/content`, kemudian mengekstraknya
ke disk lokal Colab agar loading/inference lebih stabil.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_MODEL_DIR = Path(
    "/content/drive/MyDrive/TanggapMakna/models"
)

EXPECTED_MODEL_FILES = [
    "TanggapMakna_IndoBERT_V2.zip",
    "TanggapMakna_V3A_EventClassifier.zip",
]

LOCAL_MODEL_ZIP_DIR = Path(
    "/content/tanggapmakna_locked_model_zips"
)

LOCAL_MODEL_ZIP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

uploaded_zips = []

for name in EXPECTED_MODEL_FILES:
    src = DRIVE_MODEL_DIR / name

    if not src.exists():
        raise FileNotFoundError(
            f"Model tidak ditemukan di Google Drive:\n{src}"
        )

    dst = LOCAL_MODEL_ZIP_DIR / name

    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        print("Copying:", name)
        shutil.copy2(src, dst)
    else:
        print("Already copied:", name)

    uploaded_zips.append(dst)

print("\n✅ Locked model files ready:")
for p in uploaded_zips:
    print(
        " -",
        p.name,
        f"({p.stat().st_size/(1024**2):.2f} MB)"
    )

## 3. Extract + validate ontology

In [ ]:
EXTRACT_ROOT = Path("/content/tanggapmakna_v3_fusion_models")

if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)


def extract_model_zip(zip_path, target):
    target.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(target)

    model_dirs = []

    for cfg in target.rglob("config.json"):
        parent = cfg.parent

        has_weights = (
            (parent / "model.safetensors").exists()
            or (parent / "pytorch_model.bin").exists()
        )

        if has_weights:
            model_dirs.append(parent)

    if len(model_dirs) != 1:
        raise RuntimeError(
            f"{zip_path.name}: expected 1 checkpoint directory, "
            f"ditemukan {len(model_dirs)}."
        )

    return model_dirs[0]


extracted = []

for i, zip_path in enumerate(uploaded_zips):
    model_dir = extract_model_zip(
        zip_path,
        EXTRACT_ROOT / f"model_{i}",
    )

    config = json.loads(
        (model_dir / "config.json").read_text(
            encoding="utf-8"
        )
    )

    labels = set(
        str(v)
        for v in config.get("id2label", {}).values()
    )

    extracted.append({
        "zip": zip_path,
        "dir": model_dir,
        "labels": labels,
    })


EXPECTED_SRL = {
    "O",
    "PLACE-ARG",
    "FALSE-EVENT",
    "STREET-ARG",
    "AFFECTEDOBJECTS-ARG",
    "TIME-ARG",
    "OFFICER-ARG",
    "INFORMATION-ARG",
    "FIRE-EVENT",
    "REASON-ARG",
    "ACCIDENT-EVENT",
    "DEATHVICTIM-ARG",
    "EARTHQUAKE-EVENT",
    "FLOOD-EVENT",
    "WOUNDVICTIM-ARG",
}

EXPECTED_EVENT = {
    "FLOOD",
    "EARTHQUAKE",
    "FIRE",
    "ACCIDENT",
    "NON_EVENT",
}

SRL_MODEL_DIR = None
EVENT_MODEL_DIR = None

for item in extracted:
    print("\nZIP :", item["zip"].name)
    print("DIR :", item["dir"])
    print("LABELS:", sorted(item["labels"]))

    if item["labels"] == EXPECTED_SRL:
        SRL_MODEL_DIR = item["dir"]

    elif item["labels"] == EXPECTED_EVENT:
        EVENT_MODEL_DIR = item["dir"]


if SRL_MODEL_DIR is None:
    raise RuntimeError(
        "V2 SRL checkpoint tidak ditemukan."
    )

if EVENT_MODEL_DIR is None:
    raise RuntimeError(
        "V3A baseline Event Classifier tidak ditemukan."
    )

print("\n✅ V2 SRL :", SRL_MODEL_DIR)
print("✅ V3A    :", EVENT_MODEL_DIR)

## 4. Load V2 SRL + V3A baseline

In [ ]:
srl_tokenizer = AutoTokenizer.from_pretrained(
    SRL_MODEL_DIR,
    local_files_only=True,
)

srl_model = AutoModelForTokenClassification.from_pretrained(
    SRL_MODEL_DIR,
    local_files_only=True,
)

event_tokenizer = AutoTokenizer.from_pretrained(
    EVENT_MODEL_DIR,
    local_files_only=True,
)

event_model = AutoModelForSequenceClassification.from_pretrained(
    EVENT_MODEL_DIR,
    local_files_only=True,
)

srl_model.to(DEVICE)
event_model.to(DEVICE)

srl_model.eval()
event_model.eval()


def normalize_id2label(config):
    return {
        int(k): str(v)
        for k, v in config.id2label.items()
    }


SRL_ID2LABEL = normalize_id2label(
    srl_model.config
)

EVENT_ID2LABEL = normalize_id2label(
    event_model.config
)

assert set(SRL_ID2LABEL.values()) == EXPECTED_SRL
assert set(EVENT_ID2LABEL.values()) == EXPECTED_EVENT

print("SRL model   :", srl_model.__class__.__name__)
print("Event model :", event_model.__class__.__name__)
print("✅ Selected checkpoints loaded successfully.")

# 5–14. LOCKED Fusion v1.1 implementation

Cell berikut disalin dari notebook Fusion v1.1 yang sudah lolos diagnostic QA.
**Jangan diedit selama final blind evaluation.**

## 5. Locked Fusion Configuration

In [ ]:
FUSION_CONFIG = {
    "version": "TanggapMakna_V3_Fusion_v1.1",
    "classifier_support_min_confidence": 0.60,
    "confidence_high": 0.80,
    "confidence_medium": 0.60,
    "support_roles": [
        "PLACE-ARG",
        "STREET-ARG",
        "TIME-ARG",
        "AFFECTEDOBJECTS-ARG",
        "INFORMATION-ARG",
        "REASON-ARG",
        "OFFICER-ARG",
        "DEATHVICTIM-ARG",
        "WOUNDVICTIM-ARG",
    ],
    "human_verification_required": True,
    "impact_confirmation_required": True,
    "victim_confirmation_required": True,
    "non_event_requires_false_event_support": True,
    "blind_test_locked_after_this_notebook": True,
}

print(
    json.dumps(
        FUSION_CONFIG,
        indent=2,
        ensure_ascii=False,
    )
)

## 6. V3A sentence-level inference

In [ ]:
MAX_LENGTH = 256


@torch.inference_mode()
def predict_event_classifier(text):
    encoded = event_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    encoded = {
        k: v.to(DEVICE)
        for k, v in encoded.items()
    }

    logits = event_model(**encoded).logits[0]
    probs = torch.softmax(logits, dim=-1)

    pred_id = int(probs.argmax().item())

    prob_dict = {
        EVENT_ID2LABEL[i]: float(probs[i].item())
        for i in range(len(probs))
    }

    return {
        "label": EVENT_ID2LABEL[pred_id],
        "confidence": float(probs[pred_id].item()),
        "probabilities": prob_dict,
    }

## 7. V2 SRL token-level inference

In [ ]:
TOKEN_PATTERN = re.compile(
    r"\w+(?:[-']\w+)*|[^\w\s]",
    flags=re.UNICODE,
)


def tokenize_with_offsets(text):
    result = []

    for match in TOKEN_PATTERN.finditer(text):
        result.append({
            "token": match.group(0).lower(),
            "original": match.group(0),
            "start": match.start(),
            "end": match.end(),
        })

    return result


def srl_encode_words(tokens):
    input_ids = [srl_tokenizer.cls_token_id]
    positions = []

    for word in tokens:
        pieces = srl_tokenizer.tokenize(str(word))

        if not pieces:
            pieces = [srl_tokenizer.unk_token]

        piece_ids = srl_tokenizer.convert_tokens_to_ids(pieces)

        remaining = MAX_LENGTH - 1 - len(input_ids)

        if remaining <= 0:
            break

        piece_ids = piece_ids[:remaining]
        positions.append(len(input_ids))
        input_ids.extend(piece_ids)

    input_ids.append(srl_tokenizer.sep_token_id)

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "positions": positions,
    }


@torch.inference_mode()
def predict_srl_tokens(text):
    token_items = tokenize_with_offsets(text)
    tokens = [x["token"] for x in token_items]

    encoded = srl_encode_words(tokens)

    inputs = {
        "input_ids": torch.tensor(
            [encoded["input_ids"]],
            dtype=torch.long,
            device=DEVICE,
        ),
        "attention_mask": torch.tensor(
            [encoded["attention_mask"]],
            dtype=torch.long,
            device=DEVICE,
        ),
    }

    logits = srl_model(**inputs).logits[0]
    probs = torch.softmax(logits, dim=-1)

    output = []

    for i, pos in enumerate(encoded["positions"]):
        pred_id = int(probs[pos].argmax().item())

        item = token_items[i].copy()
        item["label"] = SRL_ID2LABEL[pred_id]
        item["confidence"] = float(
            probs[pos, pred_id].item()
        )

        output.append(item)

    return output

## 8. SRL semantic spans

In [ ]:
def contiguous_role_spans(pred_tokens, role):
    spans = []
    current = []

    def flush():
        nonlocal current

        if current:
            spans.append({
                "role": role,
                "text": " ".join(
                    x["original"]
                    for x in current
                ),
                "mean_confidence": float(
                    np.mean([
                        x["confidence"]
                        for x in current
                    ])
                ),
            })

        current = []

    for item in pred_tokens:
        if item["label"] == role:
            current.append(item)
        else:
            flush()

    flush()
    return spans


def all_srl_spans(pred_tokens):
    roles = sorted(EXPECTED_SRL - {"O"})

    return {
        role: contiguous_role_spans(
            pred_tokens,
            role,
        )
        for role in roles
    }

## 9. Deterministic Time Fallback

In [ ]:
TIME_PATTERNS = [
    r"\b(?:pukul|jam)\s+\d{1,2}(?:[.:]\d{2})?(?:\s*(?:pagi|siang|sore|malam))?\b",
    r"\b(?:tadi|sejak)\s+(?:pagi|siang|sore|malam)\b",
    r"\b(?:pagi|siang|sore|malam)\s+ini\b",
    r"\bmalam\s+tadi\b",
    r"\bkemarin\b",
    r"\bbarusan\b",
    r"\btadi\b",
    r"\bhari\s+ini\b",
]


def time_fallback(text):
    matches = []
    occupied = []

    for pattern in TIME_PATTERNS:
        for m in re.finditer(
            pattern,
            text,
            flags=re.IGNORECASE,
        ):
            start, end = m.span()

            overlaps = any(
                start < e and end > s
                for s, e in occupied
            )

            if overlaps:
                continue

            matches.append({
                "role": "TIME-FALLBACK",
                "text": m.group(0),
                "source": "DETERMINISTIC_REGEX",
            })

            occupied.append((start, end))

    return sorted(
        matches,
        key=lambda x: text.lower().find(
            x["text"].lower()
        ),
    )

## 10. Victim Evidence / Negation Guard

In [ ]:

DEATH_NEGATION_PATTERNS = [
    r"\btidak\s+ada\s+(?:korban\s+)?(?:meninggal|korban\s+jiwa)\b",
    r"\btidak\s+terdapat\s+(?:korban\s+)?(?:meninggal|korban\s+jiwa)\b",
    r"\bnihil\s+(?:korban\s+)?(?:meninggal|korban\s+jiwa)\b",
    r"\btanpa\s+(?:korban\s+)?(?:meninggal|korban\s+jiwa)\b",
]

DEATH_EVIDENCE_PATTERNS = [
    r"\bmeninggal\b",
    r"\btewas\b",
    r"\bwafat\b",
    r"\bmenewaskan\b",
    r"\bkorban\s+jiwa\b",
]

WOUND_EVIDENCE_PATTERNS = [
    r"\bterluka\b",
    r"\bluka\b",
    r"\bluka\s+bakar\b",
    r"\bcedera\b",
    r"\bmengalami\s+luka\b",
    r"\bmengeluh\s+sakit\b",
]


def has_any_pattern(text, patterns):
    return any(
        re.search(pattern, text, flags=re.IGNORECASE)
        for pattern in patterns
    )


def has_death_negation(text):
    return has_any_pattern(text, DEATH_NEGATION_PATTERNS)


def normalize_victim_phrase(text, kind):
    number_word = (
        r"(?:\d+|satu|dua|tiga|empat|lima|enam|tujuh|"
        r"delapan|sembilan|sepuluh)"
    )
    person_word = r"(?:orang|korban|pengendara|penumpang|warga)"

    if kind == "wound":
        cue = (
            r"(?:terluka|cedera|luka\s+bakar|"
            r"mengalami\s+luka|mengeluh\s+sakit)"
        )
    else:
        cue = r"(?:meninggal|tewas|wafat)"

    pattern = (
        rf"\b(?:{number_word}\s+)?"
        rf"(?:{person_word}\s+)?"
        rf"{cue}\b"
    )

    match = re.search(pattern, text, flags=re.IGNORECASE)
    return match.group(0) if match else None


def apply_victim_safety(text, spans):
    safety_flags = []

    death_spans = spans.get("DEATHVICTIM-ARG", [])
    wound_spans = spans.get("WOUNDVICTIM-ARG", [])

    death_negated = has_death_negation(text)
    death_evidence = has_any_pattern(
        text,
        DEATH_EVIDENCE_PATTERNS,
    )
    wound_evidence = has_any_pattern(
        text,
        WOUND_EVIDENCE_PATTERNS,
    )

    if death_spans and death_negated:
        spans["DEATHVICTIM-ARG"] = []
        safety_flags.append(
            "DEATH_VICTIM_SUPPRESSED_BY_NEGATION"
        )

    elif death_spans and not death_evidence:
        spans["DEATHVICTIM-ARG"] = []
        safety_flags.append(
            "DEATH_VICTIM_SUPPRESSED_NO_DEATH_EVIDENCE"
        )

    elif death_spans and death_evidence:
        phrase = normalize_victim_phrase(text, "death")
        if phrase:
            spans["DEATHVICTIM-ARG"] = [{
                "role": "DEATHVICTIM-ARG",
                "text": phrase,
                "mean_confidence": float(np.mean([
                    x.get("mean_confidence", 0.0)
                    for x in death_spans
                ])),
                "source": "SRL_PLUS_LEXICAL_EVIDENCE",
            }]

    if wound_spans and not wound_evidence:
        spans["WOUNDVICTIM-ARG"] = []
        safety_flags.append(
            "WOUND_VICTIM_SUPPRESSED_NO_WOUND_EVIDENCE"
        )

    elif wound_spans and wound_evidence:
        phrase = normalize_victim_phrase(text, "wound")
        if phrase:
            spans["WOUNDVICTIM-ARG"] = [{
                "role": "WOUNDVICTIM-ARG",
                "text": phrase,
                "mean_confidence": float(np.mean([
                    x.get("mean_confidence", 0.0)
                    for x in wound_spans
                ])),
                "source": "SRL_PLUS_LEXICAL_EVIDENCE",
            }]

    return spans, safety_flags


## 11. Explicit Ambiguity / Hearsay Guard

In [ ]:

EXPLICIT_AMBIGUITY_PATTERNS = [
    r"\bbelum\s+diketahui\s+apakah\b",
    r"\bbelum\s+tahu\s+apakah\b",
    r"\bbelum\s+jelas\s+apakah\b",
    r"\bsaya\s+tidak\s+yakin\b",
    r"\btidak\s+yakin\s+(?:itu|apakah)\b",
    r"\b(?:bisa|mungkin)\s+.+?\s+atau\s+.+",
]

HEARSAY_PATTERNS = [
    r"\bkatanya\b",
    r"\bkabarnya\b",
    r"\bmenurut\s+kabar\b",
    r"\bdikabarkan\b",
]


def detect_explicit_uncertainty(text):
    reasons = []

    if any(
        re.search(p, text, flags=re.IGNORECASE)
        for p in EXPLICIT_AMBIGUITY_PATTERNS
    ):
        reasons.append("EXPLICIT_AMBIGUITY")

    if any(
        re.search(p, text, flags=re.IGNORECASE)
        for p in HEARSAY_PATTERNS
    ):
        reasons.append("HEARSAY")

    return {
        "is_uncertain": bool(reasons),
        "reasons": reasons,
    }


## 12. Fusion / Consistency Rules

In [ ]:
SRL_TRUE_EVENT_TO_CLASS = {
    "FLOOD-EVENT": "FLOOD",
    "EARTHQUAKE-EVENT": "EARTHQUAKE",
    "FIRE-EVENT": "FIRE",
    "ACCIDENT-EVENT": "ACCIDENT",
}

SUPPORT_ROLES = set(FUSION_CONFIG["support_roles"])


def confidence_band(conf):
    if conf >= FUSION_CONFIG["confidence_high"]:
        return "HIGH"

    if conf >= FUSION_CONFIG["confidence_medium"]:
        return "MEDIUM"

    return "LOW"


def derive_srl_signals(spans):
    true_events = sorted({
        SRL_TRUE_EVENT_TO_CLASS[role]
        for role in SRL_TRUE_EVENT_TO_CLASS
        if spans.get(role)
    })

    has_false = bool(spans.get("FALSE-EVENT"))

    support_present = {
        role: spans.get(role, [])
        for role in SUPPORT_ROLES
        if spans.get(role)
    }

    return {
        "true_events": true_events,
        "has_false": has_false,
        "support_present": support_present,
        "support_role_count": len(support_present),
    }


def fusion_decision(event_pred, srl_spans, uncertainty_signal=None):
    event_label = event_pred["label"]
    event_conf = event_pred["confidence"]
    signals = derive_srl_signals(srl_spans)

    true_events = signals["true_events"]
    has_false = signals["has_false"]
    support_count = signals["support_role_count"]

    result = {
        "Event_Status": "UNCERTAIN",
        "Event_Type": "UNCERTAIN",
        "Fusion_State": "UNCERTAIN",
        "Fusion_Reason": "",
        "Classifier_Label": event_label,
        "Classifier_Confidence": event_conf,
        "Classifier_Confidence_Band": confidence_band(
            event_conf
        ),
        "SRL_True_Events": true_events,
        "SRL_Has_False_Event": has_false,
        "SRL_Support_Role_Count": support_count,
        "Human_Verification_Required": True,
    }

    # Safety override: explicit uncertainty from reporter
    # outranks neural agreement.
    if uncertainty_signal and uncertainty_signal.get("is_uncertain"):
        result.update({
            "Fusion_State": "UNCERTAIN",
            "Fusion_Reason": (
                "EXPLICIT_REPORTER_UNCERTAINTY:"
                + ",".join(
                    uncertainty_signal.get("reasons", [])
                )
            ),
        })
        return result

    if (
        len(true_events) > 1
        or (true_events and has_false)
    ):
        result.update({
            "Fusion_State": "CONFLICT",
            "Fusion_Reason": "SRL_EVENT_CONFLICT",
        })
        return result

    if event_label == "NON_EVENT":
        if has_false and not true_events:
            result.update({
                "Event_Status": "FALSE_EVENT",
                "Event_Type": "NONE",
                "Fusion_State": "NON_EVENT_AGREEMENT",
                "Fusion_Reason": (
                    "V3A_NON_EVENT_AND_V2_FALSE_EVENT"
                ),
            })
            return result

        if true_events:
            result.update({
                "Fusion_State": "CONFLICT",
                "Fusion_Reason": (
                    "V3A_NON_EVENT_VS_V2_TRUE_EVENT"
                ),
            })
            return result

        result.update({
            "Fusion_State": "UNCERTAIN",
            "Fusion_Reason": (
                "CLASSIFIER_ONLY_NON_EVENT"
            ),
        })
        return result

    if true_events:
        v2_event = true_events[0]

        if v2_event == event_label:
            result.update({
                "Event_Status": "ACTUAL_EVENT",
                "Event_Type": event_label,
                "Fusion_State": "AGREEMENT_STRONG",
                "Fusion_Reason": (
                    "V3A_AND_V2_EVENT_AGREE"
                ),
            })
            return result

        result.update({
            "Fusion_State": "CONFLICT",
            "Fusion_Reason": (
                "V3A_V2_EVENT_TYPE_DISAGREE"
            ),
        })
        return result

    if has_false:
        result.update({
            "Fusion_State": "CONFLICT",
            "Fusion_Reason": (
                "V3A_EVENT_VS_V2_FALSE_EVENT"
            ),
        })
        return result

    if (
        support_count >= 1
        and event_conf
        >= FUSION_CONFIG[
            "classifier_support_min_confidence"
        ]
    ):
        result.update({
            "Event_Status": "ACTUAL_EVENT",
            "Event_Type": event_label,
            "Fusion_State": "CLASSIFIER_SUPPORTED",
            "Fusion_Reason": (
                "V3A_EVENT_WITH_SUPPORTING_SRL_ARGUMENTS"
            ),
        })
        return result

    result.update({
        "Fusion_State": "UNCERTAIN",
        "Fusion_Reason": (
            "WEAK_CLASSIFIER_ONLY_EVENT"
        ),
    })

    return result

## 13. Incident Card Builder

In [ ]:

def normalize_display_text(text):
    text = str(text)

    text = re.sub(
        r"\s+([,;:!?])",
        r"\1",
        text,
    )

    text = re.sub(
        r"(?<=\d)\s*\.\s*(?=\d)",
        ".",
        text,
    )

    text = re.sub(
        r"\s+\.",
        ".",
        text,
    )

    text = re.sub(
        r"\s{2,}",
        " ",
        text,
    )

    return text.strip()


def span_texts(spans, role):
    return [
        normalize_display_text(x["text"])
        for x in spans.get(role, [])
    ]


def build_incident_card(text, fusion, spans):
    time_values = span_texts(
        spans,
        "TIME-ARG",
    )

    time_source = "SRL"

    if not time_values:
        fallback = time_fallback(text)

        time_values = [
            normalize_display_text(x["text"])
            for x in fallback
        ]

        time_source = (
            "DETERMINISTIC_FALLBACK"
            if time_values
            else "MISSING"
        )

    return {
        "event_status": fusion["Event_Status"],
        "event_type": fusion["Event_Type"],
        "place": span_texts(spans, "PLACE-ARG"),
        "street": span_texts(spans, "STREET-ARG"),
        "time": time_values,
        "time_source": time_source,
        "affected_objects": span_texts(
            spans,
            "AFFECTEDOBJECTS-ARG",
        ),
        "information_impact": span_texts(
            spans,
            "INFORMATION-ARG",
        ),
        "reason": span_texts(spans, "REASON-ARG"),
        "officer": span_texts(spans, "OFFICER-ARG"),
        "death_victim": span_texts(
            spans,
            "DEATHVICTIM-ARG",
        ),
        "wound_victim": span_texts(
            spans,
            "WOUNDVICTIM-ARG",
        ),
        "fusion_state": fusion["Fusion_State"],
        "human_verified": False,
    }


## 14. Completeness Engine

In [ ]:
def completeness_engine(card, safety_flags):
    questions = []
    missing_fields = []

    if card["event_status"] == "FALSE_EVENT":
        return {
            "missing_fields": [],
            "adaptive_questions": [
                (
                    "Sistem mendeteksi bahwa teks mungkin bukan laporan "
                    "kejadian aktual. Apakah klasifikasi ini benar?"
                )
            ],
            "requires_human_verification": True,
            "actionability_triage": (
                "PENDING_HUMAN_VERIFICATION"
            ),
            "safety_flags": safety_flags,
        }

    if (
        card["event_status"] == "UNCERTAIN"
        or card["event_type"] == "UNCERTAIN"
    ):
        missing_fields.append("event_confirmation")
        questions.append(
            "Jenis kejadian apa yang sedang dilaporkan?"
        )

    if not (card["place"] or card["street"]):
        missing_fields.append("location")
        questions.append(
            "Di mana lokasi kejadian atau kondisi ini diamati?"
        )

    if not card["time"]:
        missing_fields.append("time")
        questions.append(
            "Kapan kejadian atau kondisi ini diamati?"
        )

    if not (
        card["affected_objects"]
        or card["information_impact"]
    ):
        missing_fields.append("impact")
        questions.append(
            "Apa dampak yang terlihat atau objek apa yang terdampak?"
        )

    if (
        card["death_victim"]
        or card["wound_victim"]
    ):
        questions.append(
            "Sistem mendeteksi informasi korban. "
            "Mohon konfirmasi apakah informasi korban tersebut benar."
        )
    else:
        missing_fields.append("victim_status")
        questions.append(
            "Apakah terdapat korban? "
            "Pilih Ya, Tidak, atau Tidak tahu."
        )

    return {
        "missing_fields": missing_fields,
        "adaptive_questions": questions,
        "requires_human_verification": True,
        "actionability_triage": (
            "PENDING_HUMAN_VERIFICATION"
        ),
        "safety_flags": safety_flags,
    }

## 15. End-to-end analyze_report()

In [ ]:
def analyze_report(text):
    event_pred = predict_event_classifier(text)
    srl_tokens = predict_srl_tokens(text)
    spans = all_srl_spans(srl_tokens)

    spans, safety_flags = apply_victim_safety(
        text,
        spans,
    )

    uncertainty_signal = detect_explicit_uncertainty(
        text
    )

    fusion = fusion_decision(
        event_pred,
        spans,
        uncertainty_signal=uncertainty_signal,
    )

    if uncertainty_signal["is_uncertain"]:
        safety_flags.append(
            "EXPLICIT_REPORTER_UNCERTAINTY"
        )

    if fusion["Fusion_State"] == "CONFLICT":
        safety_flags.append(
            "MODEL_EVENT_CONFLICT"
        )

    card = build_incident_card(
        text,
        fusion,
        spans,
    )

    completeness = completeness_engine(
        card,
        safety_flags,
    )

    return {
        "report_text": text,
        "event_classifier": event_pred,
        "srl_spans": spans,
        "fusion": fusion,
        "incident_card": card,
        "completeness": completeness,
        "srl_token_predictions": srl_tokens,
    }

# PHASE A — BLIND INFERENCE

## 16. Upload **ONLY** inference input

Upload:

`TanggapMakna_Blind_Inference_Input_v1.0.csv`

**Jangan upload ground truth pada fase ini.**

Expected SHA-256:

`6ed07fca237eb703a689f3787e409ea1917b66dca72308c4b089d5af91c7b4ac`

In [ ]:
from google.colab import files

EXPECTED_INPUT_SHA256 = (
    "6ed07fca237eb703a689f3787e409ea1917b66dca72308c4b089d5af91c7b4ac"
)

def file_sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


uploaded = files.upload()

csv_files = [
    Path("/content") / name
    for name in uploaded
    if name.lower().endswith(".csv")
]

if len(csv_files) != 1:
    raise RuntimeError(
        "Upload tepat SATU file CSV inference input."
    )

BLIND_INPUT_PATH = csv_files[0]

actual_hash = file_sha256(
    BLIND_INPUT_PATH
)

print("File   :", BLIND_INPUT_PATH.name)
print("SHA256 :", actual_hash)

if actual_hash != EXPECTED_INPUT_SHA256:
    raise RuntimeError(
        "SHA-256 inference input tidak cocok dengan file blind input LOCKED."
    )

blind_df = pd.read_csv(
    BLIND_INPUT_PATH
)

if list(blind_df.columns) != [
    "Report_ID",
    "Report_Text",
]:
    raise RuntimeError(
        "Inference input harus HANYA memiliki kolom "
        "Report_ID dan Report_Text."
    )

if len(blind_df) != 60:
    raise RuntimeError(
        f"Expected 60 reports, ditemukan {len(blind_df)}."
    )

if blind_df["Report_ID"].duplicated().any():
    raise RuntimeError(
        "Report_ID duplikat ditemukan."
    )

print("\n✅ Blind inference input LOCK verified.")
print("Rows:", len(blind_df))
display(blind_df.head())

## 17. FINAL INFERENCE — RUN ONCE

**Ini cell kritis.**

Saat dijalankan:

1. Seluruh 60 report diprediksi.
2. Hasil disimpan ke `TanggapMakna_FINAL_PREDICTIONS_FROZEN.csv`.
3. Predictions diberi SHA-256.
4. Ground truth belum digunakan.

Jika file predictions sudah ada di runtime, cell menolak overwrite untuk mencegah
rerun tidak sengaja.

In [ ]:
PREDICTION_DIR = Path(
    "/content/TanggapMakna_Final_Blind_Run"
)

PREDICTION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PREDICTION_CSV = (
    PREDICTION_DIR
    / "TanggapMakna_FINAL_PREDICTIONS_FROZEN.csv"
)

PREDICTION_JSONL = (
    PREDICTION_DIR
    / "TanggapMakna_FINAL_PREDICTIONS_FROZEN.jsonl"
)

if PREDICTION_CSV.exists():
    raise RuntimeError(
        "FROZEN prediction file sudah ada di runtime. "
        "JANGAN overwrite/rerun final inference."
    )


def join_values(values):
    if not values:
        return ""

    return "; ".join(
        str(v)
        for v in values
        if str(v).strip()
    )


flat_rows = []
full_results = []

for n, row in blind_df.iterrows():
    rid = str(row["Report_ID"])
    text = str(row["Report_Text"])

    result = analyze_report(text)

    event = result["event_classifier"]
    fusion = result["fusion"]
    card = result["incident_card"]
    comp = result["completeness"]

    full_results.append({
        "Report_ID": rid,
        **result,
    })

    flat_rows.append({
        "Report_ID": rid,
        "Report_Text": text,

        "Classifier_Label": event["label"],
        "Classifier_Confidence": event["confidence"],

        "SRL_True_Events": join_values(
            fusion["SRL_True_Events"]
        ),
        "SRL_Has_False_Event": fusion[
            "SRL_Has_False_Event"
        ],

        "Pred_Event_Status": fusion[
            "Event_Status"
        ],
        "Pred_Event_Type": fusion[
            "Event_Type"
        ],
        "Fusion_State": fusion[
            "Fusion_State"
        ],
        "Fusion_Reason": fusion[
            "Fusion_Reason"
        ],

        "Pred_Place": join_values(
            card["place"]
        ),
        "Pred_Street": join_values(
            card["street"]
        ),
        "Pred_Time": join_values(
            card["time"]
        ),
        "Pred_Affected_Objects": join_values(
            card["affected_objects"]
        ),
        "Pred_Information_Impact": join_values(
            card["information_impact"]
        ),
        "Pred_Reason": join_values(
            card["reason"]
        ),
        "Pred_Officer": join_values(
            card["officer"]
        ),
        "Pred_Death_Victim": join_values(
            card["death_victim"]
        ),
        "Pred_Wound_Victim": join_values(
            card["wound_victim"]
        ),

        "Missing_Fields": join_values(
            comp["missing_fields"]
        ),
        "Safety_Flags": join_values(
            comp["safety_flags"]
        ),
        "Adaptive_Question_Count": len(
            comp["adaptive_questions"]
        ),
    })

    if (n + 1) % 10 == 0:
        print(
            f"Processed {n + 1:02d}/60"
        )


pred_df = pd.DataFrame(flat_rows)

pred_df.to_csv(
    PREDICTION_CSV,
    index=False,
    encoding="utf-8-sig",
)

with open(
    PREDICTION_JSONL,
    "w",
    encoding="utf-8",
) as f:
    for result in full_results:
        f.write(
            json.dumps(
                result,
                ensure_ascii=False,
                default=str,
            )
            + "\n"
        )

prediction_hash = file_sha256(
    PREDICTION_CSV
)

run_manifest = {
    "project": "TanggapMakna",
    "evaluation": (
        "FINAL ONE-TIME BLIND EVALUATION"
    ),
    "fusion_version": (
        FUSION_CONFIG["version"]
    ),
    "blind_input_sha256": (
        EXPECTED_INPUT_SHA256
    ),
    "prediction_sha256": (
        prediction_hash
    ),
    "report_count": len(pred_df),
    "ground_truth_used_during_inference": False,
    "models": {
        "event_classifier": (
            "V3A baseline LOCKED"
        ),
        "srl": "V2 SRL LOCKED",
    },
    "post_test_retuning_allowed": False,
    "generated_at_utc": (
        datetime.now(timezone.utc)
        .isoformat()
    ),
}

MANIFEST_PATH = (
    PREDICTION_DIR
    / "FINAL_BLIND_RUN_MANIFEST.json"
)

MANIFEST_PATH.write_text(
    json.dumps(
        run_manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\n" + "=" * 80)
print("🔒 FINAL PREDICTIONS FROZEN")
print("=" * 80)
print("Rows              :", len(pred_df))
print("Prediction SHA256 :", prediction_hash)
print("Ground truth used : NO")
print("\nDO NOT rerun or overwrite this prediction file.")

display(
    pred_df[
        [
            "Report_ID",
            "Pred_Event_Status",
            "Pred_Event_Type",
            "Fusion_State",
            "Safety_Flags",
        ]
    ]
)

## 18. Download / preserve frozen predictions BEFORE ground truth

Download ZIP ini terlebih dahulu.

Ini adalah bukti bahwa predictions sudah dibuat dan disimpan sebelum evaluation labels dibuka.

In [ ]:
FROZEN_ARCHIVE = shutil.make_archive(
    "/content/TanggapMakna_FINAL_PREDICTIONS_FROZEN",
    "zip",
    root_dir=PREDICTION_DIR,
)

print("Frozen archive:", FROZEN_ARCHIVE)

files.download(
    FROZEN_ARCHIVE
)

# ⛔ BLINDNESS BARRIER

Pastikan:

- Cell 17 selesai.
- `TanggapMakna_FINAL_PREDICTIONS_FROZEN.zip` sudah ter-download.
- Prediction SHA-256 sudah tercetak.

**Baru setelah itu lanjut ke Phase B.**

---

# PHASE B — OPEN LOCKED GROUND TRUTH

## 19. Upload locked Ground Truth CSV

Upload:

`TanggapMakna_Blind_Ground_Truth_LOCKED_v1.0.csv`

Expected SHA-256:

`c64c3a1d8761723716acbe991450ddbe82744899afa8b7ecc7c97a5583f85acc`

In [ ]:
EXPECTED_GT_SHA256 = (
    "c64c3a1d8761723716acbe991450ddbe82744899afa8b7ecc7c97a5583f85acc"
)

if not PREDICTION_CSV.exists():
    raise RuntimeError(
        "Frozen predictions belum ada. "
        "Ground truth tidak boleh dibuka sebelum inference selesai."
    )

uploaded_gt = files.upload()

gt_csv_files = [
    Path("/content") / name
    for name in uploaded_gt
    if name.lower().endswith(".csv")
]

if len(gt_csv_files) != 1:
    raise RuntimeError(
        "Upload tepat SATU locked ground truth CSV."
    )

GT_PATH = gt_csv_files[0]

gt_hash = file_sha256(GT_PATH)

print("Ground truth file:", GT_PATH.name)
print("SHA256           :", gt_hash)

if gt_hash != EXPECTED_GT_SHA256:
    raise RuntimeError(
        "SHA-256 ground truth tidak cocok dengan LOCKED ground truth."
    )

gt_df = pd.read_csv(
    GT_PATH
)

required_gt_cols = [
    "Report_ID",
    "Report_Text",
    "Final_Event_Status",
    "Final_Event_Type",
    "Final_Place",
    "Final_Street",
    "Final_Time",
    "Final_Affected_Objects",
    "Final_Information_Impact",
    "Final_Reason",
    "Final_Officer",
    "Final_Death_Victim",
    "Final_Wound_Victim",
]

missing = [
    c
    for c in required_gt_cols
    if c not in gt_df.columns
]

if missing:
    raise RuntimeError(
        f"Ground truth columns missing: {missing}"
    )

if len(gt_df) != 60:
    raise RuntimeError(
        f"Expected 60 GT rows, got {len(gt_df)}."
    )

pred_ids = pred_df[
    "Report_ID"
].astype(str).tolist()

gt_ids = gt_df[
    "Report_ID"
].astype(str).tolist()

if pred_ids != gt_ids:
    raise RuntimeError(
        "Report_ID/order mismatch between predictions and GT."
    )

print("\n✅ LOCKED Ground Truth opened AFTER predictions were frozen.")

## 20. Metric definitions — frozen before calculation

### Classification

**Event Status Macro-F1**
- classes: `ACTUAL_EVENT / FALSE_EVENT / UNCERTAIN`

**Event Type Macro-F1**
- evaluated on **GT ACTUAL_EVENT only**
- target classes: `FLOOD / EARTHQUAKE / FIRE / ACCIDENT`
- if Fusion abstains or outputs non-actual status, prediction is treated as abstention and contributes false negatives.

**False-event F1**
- binary `FALSE_EVENT` vs `NOT_FALSE_EVENT`.

### Semantic extraction

Token-level micro Precision/Recall/F1:
- Location = `Place + Street`
- Time = `Time`
- Impact = `Affected_Objects + Information_Impact`
- Victim = `Death_Victim + Wound_Victim`

### Product / safety

**Product Correct**
- status must match GT;
- if GT is `ACTUAL_EVENT`, event type must also match.

**Safe Abstention**
- Fusion outputs `UNCERTAIN` while GT is ACTUAL/FALSE.
- Classification miss, but system avoids an incorrect automatic confirmation.

**Unsafe Confirmation**
Any of:
- GT `UNCERTAIN`, prediction is `ACTUAL_EVENT` or `FALSE_EVENT`;
- GT `ACTUAL_EVENT`, prediction is `FALSE_EVENT`;
- GT `FALSE_EVENT`, prediction is `ACTUAL_EVENT`;
- GT `ACTUAL_EVENT`, prediction is `ACTUAL_EVENT` with wrong event type.

**Auto-decision coverage**
- fraction with predicted status other than `UNCERTAIN`.

**Core Human Intervention**
- Fusion abstains (`UNCERTAIN`) or product decision is wrong.

Human verification itself remains mandatory for **all** reports by system design.

In [ ]:
STATUS_LABELS = [
    "ACTUAL_EVENT",
    "FALSE_EVENT",
    "UNCERTAIN",
]

EVENT_TYPES = [
    "FLOOD",
    "EARTHQUAKE",
    "FIRE",
    "ACCIDENT",
]


def clean_value(v):
    if pd.isna(v):
        return ""
    return str(v).strip()


def normalize_tokens(value):
    text = clean_value(value).casefold()

    if not text:
        return []

    # Treat clock punctuation equivalently.
    text = re.sub(
        r"(?<=\d)[.:](?=\d)",
        " ",
        text,
    )

    return re.findall(
        r"\w+",
        text,
        flags=re.UNICODE,
    )


def combine_fields(df, columns):
    result = []

    for _, row in df.iterrows():
        values = [
            clean_value(row[c])
            for c in columns
            if clean_value(row[c])
        ]
        result.append(
            "; ".join(values)
        )

    return result


def micro_token_metrics(gt_values, pred_values):
    tp = 0
    gt_total = 0
    pred_total = 0
    exact = 0

    for gt_val, pred_val in zip(
        gt_values,
        pred_values,
    ):
        gt_tokens = Counter(
            normalize_tokens(gt_val)
        )
        pred_tokens = Counter(
            normalize_tokens(pred_val)
        )

        tp += sum(
            (gt_tokens & pred_tokens)
            .values()
        )

        gt_total += sum(
            gt_tokens.values()
        )

        pred_total += sum(
            pred_tokens.values()
        )

        if gt_tokens == pred_tokens:
            exact += 1

    precision = (
        tp / pred_total
        if pred_total
        else (
            1.0
            if gt_total == 0
            else 0.0
        )
    )

    recall = (
        tp / gt_total
        if gt_total
        else 1.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall
        else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "exact_report_rate": (
            exact / len(gt_values)
        ),
        "tp_tokens": tp,
        "gt_tokens": gt_total,
        "pred_tokens": pred_total,
    }


eval_df = gt_df.merge(
    pred_df,
    on=[
        "Report_ID",
        "Report_Text",
    ],
    how="inner",
    validate="one_to_one",
)

assert len(eval_df) == 60

gt_status = eval_df[
    "Final_Event_Status"
].astype(str)

pred_status = eval_df[
    "Pred_Event_Status"
].astype(str)

gt_type = eval_df[
    "Final_Event_Type"
].astype(str)

pred_type = eval_df[
    "Pred_Event_Type"
].astype(str)


# ------------------------------------------------------------
# 1. Event Status
# ------------------------------------------------------------
status_accuracy = accuracy_score(
    gt_status,
    pred_status,
)

status_macro_f1 = f1_score(
    gt_status,
    pred_status,
    labels=STATUS_LABELS,
    average="macro",
    zero_division=0,
)

status_report = classification_report(
    gt_status,
    pred_status,
    labels=STATUS_LABELS,
    output_dict=True,
    zero_division=0,
)

status_cm = confusion_matrix(
    gt_status,
    pred_status,
    labels=STATUS_LABELS,
)


# ------------------------------------------------------------
# 2. Event Type on GT ACTUAL_EVENT only
# ------------------------------------------------------------
actual_mask = (
    gt_status == "ACTUAL_EVENT"
)

actual_gt_type = gt_type[
    actual_mask
].reset_index(drop=True)

actual_pred_type = []

for _, row in eval_df[
    actual_mask
].iterrows():
    if (
        row["Pred_Event_Status"]
        == "ACTUAL_EVENT"
        and row["Pred_Event_Type"]
        in EVENT_TYPES
    ):
        actual_pred_type.append(
            row["Pred_Event_Type"]
        )
    else:
        actual_pred_type.append(
            "ABSTAIN_OR_NON_ACTUAL"
        )

type_macro_f1_actual = f1_score(
    actual_gt_type,
    actual_pred_type,
    labels=EVENT_TYPES,
    average="macro",
    zero_division=0,
)

accepted_actual_mask = (
    actual_mask
    & (
        pred_status
        == "ACTUAL_EVENT"
    )
)

accepted_actual_count = int(
    accepted_actual_mask.sum()
)

actual_count = int(
    actual_mask.sum()
)

accepted_correct_type = int(
    (
        accepted_actual_mask
        & (
            gt_type
            == pred_type
        )
    ).sum()
)

actual_auto_coverage = (
    accepted_actual_count
    / actual_count
    if actual_count
    else 0.0
)

accepted_type_accuracy = (
    accepted_correct_type
    / accepted_actual_count
    if accepted_actual_count
    else 0.0
)


# ------------------------------------------------------------
# 3. False-event binary
# ------------------------------------------------------------
gt_false = (
    gt_status == "FALSE_EVENT"
).astype(int)

pred_false = (
    pred_status == "FALSE_EVENT"
).astype(int)

false_p, false_r, false_f1, _ = (
    precision_recall_fscore_support(
        gt_false,
        pred_false,
        average="binary",
        zero_division=0,
    )
)


# ------------------------------------------------------------
# 4. Semantic extraction
# ------------------------------------------------------------
gt_location = combine_fields(
    eval_df,
    [
        "Final_Place",
        "Final_Street",
    ],
)

pred_location = combine_fields(
    eval_df,
    [
        "Pred_Place",
        "Pred_Street",
    ],
)

gt_time = combine_fields(
    eval_df,
    ["Final_Time"],
)

pred_time = combine_fields(
    eval_df,
    ["Pred_Time"],
)

gt_impact = combine_fields(
    eval_df,
    [
        "Final_Affected_Objects",
        "Final_Information_Impact",
    ],
)

pred_impact = combine_fields(
    eval_df,
    [
        "Pred_Affected_Objects",
        "Pred_Information_Impact",
    ],
)

gt_victim = combine_fields(
    eval_df,
    [
        "Final_Death_Victim",
        "Final_Wound_Victim",
    ],
)

pred_victim = combine_fields(
    eval_df,
    [
        "Pred_Death_Victim",
        "Pred_Wound_Victim",
    ],
)

location_metrics = micro_token_metrics(
    gt_location,
    pred_location,
)

time_metrics = micro_token_metrics(
    gt_time,
    pred_time,
)

impact_metrics = micro_token_metrics(
    gt_impact,
    pred_impact,
)

victim_metrics = micro_token_metrics(
    gt_victim,
    pred_victim,
)


# ------------------------------------------------------------
# 5. Product correctness / safety
# ------------------------------------------------------------
status_correct = (
    gt_status
    == pred_status
)

type_correct_if_actual = (
    (~actual_mask)
    | (
        gt_type
        == pred_type
    )
)

product_correct = (
    status_correct
    & type_correct_if_actual
)

safe_abstention = (
    (
        pred_status
        == "UNCERTAIN"
    )
    & (
        gt_status
        != "UNCERTAIN"
    )
)

unsafe_status_confirmation = (
    (
        pred_status
        != "UNCERTAIN"
    )
    & (
        pred_status
        != gt_status
    )
)

unsafe_wrong_type_confirmation = (
    (
        gt_status
        == "ACTUAL_EVENT"
    )
    & (
        pred_status
        == "ACTUAL_EVENT"
    )
    & (
        gt_type
        != pred_type
    )
)

unsafe_confirmation = (
    unsafe_status_confirmation
    | unsafe_wrong_type_confirmation
)

auto_decision = (
    pred_status
    != "UNCERTAIN"
)

correct_uncertain_handling = (
    (
        gt_status
        == "UNCERTAIN"
    )
    & (
        pred_status
        == "UNCERTAIN"
    )
)

gt_uncertain_count = int(
    (
        gt_status
        == "UNCERTAIN"
    ).sum()
)

status_error = ~status_correct

core_human_intervention = (
    (
        pred_status
        == "UNCERTAIN"
    )
    | (
        ~product_correct
    )
)


# ------------------------------------------------------------
# Summary metrics
# ------------------------------------------------------------
summary_metrics = {
    "report_count": len(eval_df),

    "event_status_accuracy": status_accuracy,
    "event_status_macro_f1": status_macro_f1,

    "event_type_macro_f1_on_gt_actual": (
        type_macro_f1_actual
    ),
    "gt_actual_count": actual_count,
    "actual_auto_decision_coverage": (
        actual_auto_coverage
    ),
    "accepted_actual_type_accuracy": (
        accepted_type_accuracy
    ),

    "false_event_precision": false_p,
    "false_event_recall": false_r,
    "false_event_f1": false_f1,

    "location_token_precision": (
        location_metrics["precision"]
    ),
    "location_token_recall": (
        location_metrics["recall"]
    ),
    "location_token_f1": (
        location_metrics["f1"]
    ),

    "time_token_precision": (
        time_metrics["precision"]
    ),
    "time_token_recall": (
        time_metrics["recall"]
    ),
    "time_token_f1": (
        time_metrics["f1"]
    ),

    "impact_token_precision": (
        impact_metrics["precision"]
    ),
    "impact_token_recall": (
        impact_metrics["recall"]
    ),
    "impact_token_f1": (
        impact_metrics["f1"]
    ),

    "victim_token_precision": (
        victim_metrics["precision"]
    ),
    "victim_token_recall": (
        victim_metrics["recall"]
    ),
    "victim_token_f1": (
        victim_metrics["f1"]
    ),

    "product_correct_rate": float(
        product_correct.mean()
    ),

    "auto_decision_coverage": float(
        auto_decision.mean()
    ),

    "auto_decision_product_accuracy": (
        float(
            product_correct[
                auto_decision
            ].mean()
        )
        if auto_decision.any()
        else 0.0
    ),

    "safe_abstention_count": int(
        safe_abstention.sum()
    ),
    "safe_abstention_rate_all": float(
        safe_abstention.mean()
    ),

    "safe_abstention_share_of_status_errors": (
        float(
            safe_abstention.sum()
            / status_error.sum()
        )
        if status_error.sum()
        else 0.0
    ),

    "unsafe_confirmation_count": int(
        unsafe_confirmation.sum()
    ),
    "unsafe_confirmation_rate_all": float(
        unsafe_confirmation.mean()
    ),

    "correct_uncertain_handling_rate": (
        float(
            correct_uncertain_handling.sum()
            / gt_uncertain_count
        )
        if gt_uncertain_count
        else 0.0
    ),

    "core_human_intervention_rate": float(
        core_human_intervention.mean()
    ),

    "fusion_conflict_count": int(
        (
            eval_df["Fusion_State"]
            == "CONFLICT"
        ).sum()
    ),
}


print("=" * 80)
print("TANGGAPMAKNA — FINAL BLIND METRICS")
print("=" * 80)

for key, value in summary_metrics.items():
    if isinstance(value, float):
        print(
            f"{key:<46}: {value:.4f}"
        )
    else:
        print(
            f"{key:<46}: {value}"
        )

## 21. Event Status confusion matrix + per-class report

In [ ]:
status_cm_df = pd.DataFrame(
    status_cm,
    index=[
        f"GT_{x}"
        for x in STATUS_LABELS
    ],
    columns=[
        f"PRED_{x}"
        for x in STATUS_LABELS
    ],
)

display(status_cm_df)

status_report_df = (
    pd.DataFrame(
        status_report
    )
    .T
)

display(status_report_df)

## 22. Per-report safety/error audit

Gunakan tabel ini untuk melihat **hasil final**, bukan untuk melakukan tuning.

In [ ]:
eval_df["Status_Correct"] = (
    status_correct.values
)

eval_df[
    "Type_Correct_If_GT_Actual"
] = (
    type_correct_if_actual.values
)

eval_df["Product_Correct"] = (
    product_correct.values
)

eval_df["Safe_Abstention"] = (
    safe_abstention.values
)

eval_df["Unsafe_Confirmation"] = (
    unsafe_confirmation.values
)

eval_df["Core_Human_Intervention"] = (
    core_human_intervention.values
)

audit_cols = [
    "Report_ID",
    "Report_Text",
    "Final_Event_Status",
    "Final_Event_Type",
    "Pred_Event_Status",
    "Pred_Event_Type",
    "Fusion_State",
    "Fusion_Reason",
    "Status_Correct",
    "Type_Correct_If_GT_Actual",
    "Product_Correct",
    "Safe_Abstention",
    "Unsafe_Confirmation",
    "Safety_Flags",
]

display(
    eval_df[audit_cols]
)

print("\nUnsafe confirmations:")
display(
    eval_df.loc[
        eval_df[
            "Unsafe_Confirmation"
        ],
        audit_cols,
    ]
)

print("\nSafe abstentions:")
display(
    eval_df.loc[
        eval_df[
            "Safe_Abstention"
        ],
        audit_cols,
    ]
)

## 23. Save FINAL evaluation artifacts

Output:
- final metrics CSV + JSON
- status confusion matrix
- per-report evaluation
- frozen predictions
- manifest
- single ZIP for submission back to ChatGPT

**Tidak ada automatic pass/fail dan tidak ada retuning.**

In [ ]:
RESULT_DIR = Path(
    "/content/TanggapMakna_FINAL_EVALUATION_RESULTS"
)

if RESULT_DIR.exists():
    shutil.rmtree(
        RESULT_DIR
    )

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Metrics table
metrics_rows = []

for key, value in summary_metrics.items():
    metrics_rows.append({
        "Metric": key,
        "Value": value,
    })

metrics_df = pd.DataFrame(
    metrics_rows
)

metrics_df.to_csv(
    RESULT_DIR
    / "Final_Metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

(
    RESULT_DIR
    / "Final_Metrics.json"
).write_text(
    json.dumps(
        summary_metrics,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

status_cm_df.to_csv(
    RESULT_DIR
    / "Event_Status_Confusion_Matrix.csv",
    encoding="utf-8-sig",
)

status_report_df.to_csv(
    RESULT_DIR
    / "Event_Status_Per_Class.csv",
    encoding="utf-8-sig",
)

eval_df.to_csv(
    RESULT_DIR
    / "Final_Per_Report_Evaluation.csv",
    index=False,
    encoding="utf-8-sig",
)

# Preserve original frozen artifacts.
shutil.copy2(
    PREDICTION_CSV,
    RESULT_DIR
    / PREDICTION_CSV.name,
)

shutil.copy2(
    PREDICTION_JSONL,
    RESULT_DIR
    / PREDICTION_JSONL.name,
)

shutil.copy2(
    MANIFEST_PATH,
    RESULT_DIR
    / MANIFEST_PATH.name,
)

final_manifest = {
    **run_manifest,
    "locked_ground_truth_sha256": (
        EXPECTED_GT_SHA256
    ),
    "evaluation_completed": True,
    "summary_metrics": summary_metrics,
    "rules_changed_after_blind_test": False,
    "final_ai_status": (
        "FREEZE AFTER DOCUMENTING RESULTS"
    ),
}

(
    RESULT_DIR
    / "FINAL_EVALUATION_MANIFEST.json"
).write_text(
    json.dumps(
        final_manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

FINAL_ARCHIVE = shutil.make_archive(
    "/content/TanggapMakna_FINAL_BLIND_EVALUATION_RESULTS",
    "zip",
    root_dir=RESULT_DIR,
)

print("=" * 80)
print("✅ FINAL BLIND EVALUATION COMPLETE")
print("=" * 80)
print("Archive:", FINAL_ARCHIVE)
print(
    "\nIMPORTANT: "
    "Do not modify V3 Fusion v1.1 using these test results."
)

files.download(
    FINAL_ARCHIVE
)

# FINAL CHECKPOINT

Setelah notebook selesai, kirim ke ChatGPT:

`TanggapMakna_FINAL_BLIND_EVALUATION_RESULTS.zip`

Jangan mengirim hanya screenshot metrik karena ZIP menyimpan:
- prediction audit trail,
- per-report outputs,
- confusion matrix,
- final metrics,
- hashes/manifests.

Setelah hasil ini dianalisis:

**AI FREEZE → Django implementation.**